In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity,euclidean_distances

In [84]:
df = pd.read_csv(r"C:\Users\Siddarood\Data Science 2\DS Morning 9th Feb 2026\DS Morning 9th Feb 2026\Movie.csv")

In [85]:
df.shape

(8992, 3)

In [86]:
df.head()

,userId,movie,rating
0,3,Toy Story (1995),4.0
1,6,Toy Story (1995),5.0
2,8,Toy Story (1995),4.0
3,10,Toy Story (1995),4.0
4,11,Toy Story (1995),4.5


* So we have data of 8,992 rows data with the movie name and the rating

In [87]:
df.duplicated().sum()

np.int64(0)

In [88]:
df.isnull().sum()

userId    0
movie     0
rating    0
dtype: int64

* Our Agend Here is to build a recommnedation model based on Item-Itme & User-to-User Similarity

In [89]:
len(df.userId.unique())

4081

* The rowsize is 8,992 but here the same user have may seen the many different types of movies.
* So to know the how many user are present we are using the unique with len funtion so it will let us now the how many users we have in total.

In [90]:
len(df.movie.unique())

10

* So the data set only contain only 10 unique movies

In [91]:
len(df.rating.unique())
df.rating.value_counts()

rating
3.0    2736
4.0    2660
5.0    1394
3.5     679
2.0     542
4.5     374
2.5     277
1.0     212
1.5      61
0.5      57
Name: count, dtype: int64

* We have 10 types of diffrent ratings.
* So the rating value count is showing which rating is more given by user data set i.e is "rating -> 3.0 (2,36)" and "rating-> 4.0 (2,660)",....

In [92]:
df.movie.unique()

array(['Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)',
       'Waiting to Exhale (1995)', 'Father of the Bride Part II (1995)',
       'Heat (1995)', 'Sabrina (1995)', 'Tom and Huck (1995)',
       'Sudden Death (1995)', 'GoldenEye (1995)'], dtype=object)

In [93]:
df.groupby('movie')['rating'].mean().sort_values(ascending=False)

movie
Toy Story (1995)                      3.959323
Heat (1995)                           3.836508
GoldenEye (1995)                      3.427003
Sabrina (1995)                        3.381429
Tom and Huck (1995)                   3.352564
Jumanji (1995)                        3.268398
Grumpier Old Men (1995)               3.186861
Father of the Bride Part II (1995)    3.143836
Sudden Death (1995)                   3.017327
Waiting to Exhale (1995)              3.000000
Name: rating, dtype: float64

* Using the groupby We are checking which movie we got the highest no of ratings. 

## Collabrative Based : Item-Item
   ### Cosine Similarity

* To use Cosine Similarity Our Data should be in the pivot table foramt.
* Note: The cosine similarity take data with respect to row.

In [94]:
df1 = df.pivot_table(index='movie', columns='userId', values='rating')
df1.fillna(0,inplace=True)
df1.head()

userId,1,2,3,4,5,6,7,8,10,11,...,7105,7107,7108,7110,7113,7115,7116,7117,7119,7120
movie,,,,,,,,,,,,,,,,,,,,,
Father of the Bride Part II (1995),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,3.5,0.0,0.0,0.0
GoldenEye (1995),0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,2.5,...,2.0,0.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0
Grumpier Old Men (1995),0.0,4.0,0.0,0.0,0.0,3.0,3.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
Heat (1995),0.0,0.0,0.0,3.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,5.0,0.0,0.0
Jumanji (1995),3.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,4.0


* We are working with item-item so with the movie index we are calculating cosine similraity.
* while using the movie index where movie rating is not present the data will be given as nan so to avoid the missing value and for the calculatin we are filling the missing values with the 0.

cos_sim = cosine_similarity(df1)
cos_sim

In [95]:
cos_sim = cosine_similarity(df1)
cos_sim.shape

(10, 10)

In [96]:
cos_sim[0]

array([1.        , 0.21599275, 0.45566216, 0.24736744, 0.27980393,
       0.39574604, 0.22645759, 0.16293622, 0.30863759, 0.16378738])

* We have total 10 movie, The 10 scores are given for the all the movies.
* The first row and first score 1, is cos_similarity btw with itself and the second score is first movie similarity score with second movie like wise for all the movies the cosaine_similarity score is calculated with the first movie.
* Like wise the second rows also the score btw it self and with all other 10 movies like wise all the movies similarity score is calculated with it self and others 9 movies.

In [97]:
df1.index

Index(['Father of the Bride Part II (1995)', 'GoldenEye (1995)',
       'Grumpier Old Men (1995)', 'Heat (1995)', 'Jumanji (1995)',
       'Sabrina (1995)', 'Sudden Death (1995)', 'Tom and Huck (1995)',
       'Toy Story (1995)', 'Waiting to Exhale (1995)'],
      dtype='object', name='movie')

In [98]:
np.where('Grumpier Old Men (1995)'==df1.index)[0][0]
# This will return the index value of the movie we are serching.

np.int64(2)

In [99]:
def recommended_movie(movie_name):
    if movie_name in df1.index:
        index= np.where(movie_name==df1.index)[0][0]
        similar= sorted(list(enumerate(cos_sim[index])),reverse=True,key=lambda x: x[1])[1:6]
        print(f'Recommended_movie for{movie_name}')
        print('-'*40)
        for movie in similar:
            print(df1.index[movie[0]])
    else:
        print('Movie is Not in the List')

In [100]:
recommended_movie('Jumanji (1995)')

Recommended_movie forJumanji (1995)
----------------------------------------
Toy Story (1995)
GoldenEye (1995)
Heat (1995)
Father of the Bride Part II (1995)
Sabrina (1995)


* In above code we are giving the top 5 recommended movie for the viewer's based highest similarity score calculated.
* First step with if condition we are checking is the movie name present in the given list if yes it will go further else it will give the other condtion and give the ouput movie name is not in the list.
* step 1:"index=np.where(movie_name==df1.index)[0][0]" * check the movie and give the index postion of that movie.
*  step 2: "similar= sorted(list(enumerate(cos_sim[index])),reverse=True,key=lambda x: x[1])[1:6]" with respect to similarity score in sorted in revres (with hihgest score) through lambda function and selecting first 5 movies.
*  So we need Movie Name as out put for that used the for looping "for movie in similar:
print(df1.index[movie[0]])" throuhg looping wiht respect to index wt the movive name is there we are printing that output.

### Euclidean_distance

In [101]:
euc_dist = euclidean_distances(df1)
euc_dist[0]
# This is the euclidean_distances

array([  0.        , 146.49658699,  89.65628812, 144.20384877,
       122.72224737,  97.89662916,  86.34813258,  85.25549836,
       196.5362562 ,  86.69919261])

In [102]:
def recommended_movie_euc(movie_name):
    if movie_name in df1.index:
        index= np.where(movie_name==df1.index)[0][0]
        similar= sorted(list(enumerate(euc_dist[index])),reverse=False,key=lambda x: x[1])[1:6]
        print(f'Recommended_movie for{movie_name}')
        print('-'*40)
        for movie in similar:
            print(df1.index[movie[0]])
    else:
        print('Movie is Not in the List')

In [103]:
recommended_movie_euc('GoldenEye (1995)')

Recommended_movie forGoldenEye (1995)
----------------------------------------
Tom and Huck (1995)
Waiting to Exhale (1995)
Sudden Death (1995)
Jumanji (1995)
Father of the Bride Part II (1995)


* This are the Recommended Movies using the Euclidean Distance
* Here we are given revrse==False Because we need the less distance to consider the user have similart taste.

### Pearson Correlation

Note: The Pearson Correlation find Relation column with column.

In [104]:
df1 = df.pivot_table(index='userId', columns='movie', values='rating')
df1.fillna(0,inplace=True)
df1.head()

movie,Father of the Bride Part II (1995),GoldenEye (1995),Grumpier Old Men (1995),Heat (1995),Jumanji (1995),Sabrina (1995),Sudden Death (1995),Tom and Huck (1995),Toy Story (1995),Waiting to Exhale (1995)
userId,,,,,,,,,,
1,0.0,0.0,0.0,0.0,3.5,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
4,0.0,4.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0


In [105]:
df1_p = df1.corr(method='pearson')
pearson=df1_p.values
pearson

array([[ 1.        , -0.01592626,  0.35981026,  0.05182799,  0.1065231 ,
         0.28734257,  0.1607488 ,  0.12267294,  0.02151568,  0.10682335],
       [-0.01592626,  1.        , -0.02117329,  0.11022266,  0.14131531,
        -0.02194024,  0.03609683,  0.05125026, -0.15539343,  0.03099449],
       [ 0.35981026, -0.02117329,  1.        ,  0.10032981,  0.05610614,
         0.27729779,  0.19128165,  0.1009599 ,  0.0223755 ,  0.09726919],
       [ 0.05182799,  0.11022266,  0.10032981,  1.        ,  0.00415785,
         0.05973991,  0.17078777, -0.01071283, -0.03805236,  0.03566654],
       [ 0.1065231 ,  0.14131531,  0.05610614,  0.00415785,  1.        ,
         0.05465811,  0.04110294,  0.15109003,  0.02559326,  0.0692512 ],
       [ 0.28734257, -0.02194024,  0.27729779,  0.05973991,  0.05465811,
         1.        ,  0.13028067,  0.12308716,  0.00279841,  0.13283779],
       [ 0.1607488 ,  0.03609683,  0.19128165,  0.17078777,  0.04110294,
         0.13028067,  1.        ,  0.07128401

* So to use the pearson we need code which support the dataframe or array our code is ready to work with the array so we are chainging the dataframe to array and using to find the values.

In [108]:
df1.index

Index([   1,    2,    3,    4,    5,    6,    7,    8,   10,   11,
       ...
       7105, 7107, 7108, 7110, 7113, 7115, 7116, 7117, 7119, 7120],
      dtype='int64', name='userId', length=4081)

In [106]:
def recommended_movie_p(movie_name):
    if movie_name in df1.columns:
        index= np.where(movie_name==df1.columns)[0][0]
        similar= sorted(list(enumerate(pearson[index])),reverse=False,key=lambda x: x[1])[1:6]
        print(f'Recommended_movie for{movie_name}')
        print('-'*40)
        for movie in similar:
            print(df1.columns[movie[0]])
    else:
        print('Movie is Not in the List')

In [107]:
recommended_movie_p('Grumpier Old Men (1995)')

Recommended_movie forGrumpier Old Men (1995)
----------------------------------------
Toy Story (1995)
Jumanji (1995)
Waiting to Exhale (1995)
Heat (1995)
Tom and Huck (1995)


* Now our data set row contains arrya and columns contain movie names so we should change here df1.index to df1.columns. Because the pearson work with column format.

### User-User:
  #### Cosine_similarity

In [109]:
df2 = df.pivot_table(index='userId',columns='movie',values='rating')
df2.fillna(0,inplace=True)
df2.head()

movie,Father of the Bride Part II (1995),GoldenEye (1995),Grumpier Old Men (1995),Heat (1995),Jumanji (1995),Sabrina (1995),Sudden Death (1995),Tom and Huck (1995),Toy Story (1995),Waiting to Exhale (1995)
userId,,,,,,,,,,
1,0.0,0.0,0.0,0.0,3.5,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
4,0.0,4.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0


In [110]:
df3=pd.DataFrame(cosine_similarity(df2),columns=df.userId.unique(),index=df.userId.unique())
df3.head()

,3,6,8,10,11,12,13,14,16,19,...,6975,6979,6993,7030,7031,7044,7070,7080,7087,7105
3,1.0,0.0,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,1.0,0.707107,0.000000,0.0,0.000000,0.000000,0.0,0.553372
6,0.0,1.0,0.0,0.0,0.0,0.390567,0.707107,0.615457,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.458831,0.0,0.000000
8,0.0,0.0,1.0,0.0,0.0,0.650945,0.000000,0.492366,1.0,0.874157,...,0.0,1.0,0.0,0.707107,0.000000,0.0,0.752577,0.458831,1.0,0.622543
10,0.0,0.0,0.0,1.0,0.0,0.000000,0.000000,0.615457,0.0,0.388514,...,0.8,0.0,0.0,0.000000,0.989949,0.0,0.000000,0.619422,0.0,0.000000
11,1.0,0.0,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,1.0,0.707107,0.000000,0.0,0.000000,0.000000,0.0,0.553372


* The output is showing simimlarity score for all users,so the column and row contain the similarity score of the users with eachother.
* So how to read data see first row data and column its the similairty score with itself. below in the score in the second index is cosine_similarity score with 3 & 6 below that 3 & 8. so like waise each row and column contain score btw itself and all other users.

In [111]:
user = 14
thr =0.9

In [112]:
df3[df3[user]>thr][user].sort_values(ascending=False)[1:11]

4138    0.995703
987     0.992625
3505    0.972373
6318    0.968856
2005    0.959475
3609    0.957427
2607    0.947705
2890    0.946843
5628    0.945484
3142    0.938083
Name: 14, dtype: float64

* In the above code we are seeing the similarity score of the user 14 & keept the threshold 0.9. So based on the simiarity score other user person having the score above 0.9 is showing the output of top 10 users.

In [113]:
df[(df.userId==14) | (df.userId==4138)]

,userId,movie,rating
7,14,Toy Story (1995),4.5
3251,4138,Jumanji (1995),4.0
5948,4138,Heat (1995),3.0
6467,14,Sabrina (1995),3.5
8362,4138,GoldenEye (1995),3.0


* To See the user 'x' and 'y' what movies the each user have watched and what rating they have given and what movie can be suggested to each of them to check that based on the cosine_similarity_score and thershold we are cross checking wt to suggest them.
* user 14 watched 'toy story & sabrina' and user 4138 have watched 'Jumanji,Heat & GoldenEye' means these to user have the similar taste so the user 14 movies can be suggested user 4138 and his movies can be suggested to 14.
* How We know that This two have similar Taste?
*  Ans:- The similarity score have calculated by in each category what movies they have seen and to those movie what rating they have given based on that the cosaine_similarity is calculated an for that we kept a threshold 0.9 if its corssed that we can consider these two have same taste.

In [114]:
Euc_dis_df2 = euclidean_distances(df2)
Euc_dis_df2[0]

array([0.        , 5.31507291, 5.31507291, ..., 9.39414711, 6.10327781,
       6.04152299], shape=(4081,))

In [115]:
df3_e = pd.DataFrame(euclidean_distances(df2),columns=df.userId.unique(),index=df.userId.unique())
df3_e.head()

,3,6,8,10,11,12,13,14,16,19,...,6975,6979,6993,7030,7031,7044,7070,7080,7087,7105
3,0.000000,5.315073,5.315073,6.103278,0.500000,8.440972,5.500000,8.845903,5.315073,6.224950,...,4.031129,5.315073,1.500000,4.031129,5.500000,5.315073,6.363961,9.394147,6.103278,6.041523
6,5.315073,0.000000,5.656854,6.403124,5.000000,7.141428,3.162278,6.480741,5.656854,6.519202,...,4.472136,5.656854,4.472136,6.928203,5.830952,5.656854,6.652067,7.745967,6.403124,8.261356
8,5.315073,5.656854,0.000000,6.403124,5.000000,5.916080,5.830952,7.071068,0.000000,2.549510,...,4.472136,0.000000,4.472136,4.000000,5.830952,5.656854,3.500000,7.745967,1.000000,5.678908
10,6.103278,6.403124,6.403124,0.000000,5.830952,9.165151,6.557439,6.403124,6.403124,5.612486,...,3.605551,6.403124,5.385165,7.549834,1.000000,6.403124,7.297260,6.855655,7.071068,8.789198
11,0.500000,5.000000,5.000000,5.830952,0.000000,8.246211,5.196152,8.660254,5.000000,5.958188,...,3.605551,5.000000,1.000000,4.123106,5.196152,5.000000,6.103278,9.219544,5.830952,6.103278


In [116]:
user = 14 
thr = 10

In [117]:
df3_e[df3_e[user]<thr][user].sort_values()[1:11]

4138    1.936492
3505    1.936492
6318    2.061553
2005    2.291288
3609    2.449490
987     2.500000
2607    2.598076
5628    2.645751
2890    2.645751
6241    3.000000
Name: 14, dtype: float64

### Pearson_Correlation

In [118]:
df2 = df.pivot_table(index='movie',columns='userId',values='rating')
df2.fillna(0,inplace=True)
df2.head()

userId,1,2,3,4,5,6,7,8,10,11,...,7105,7107,7108,7110,7113,7115,7116,7117,7119,7120
movie,,,,,,,,,,,,,,,,,,,,,
Father of the Bride Part II (1995),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,3.5,0.0,0.0,0.0
GoldenEye (1995),0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,2.5,...,2.0,0.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0
Grumpier Old Men (1995),0.0,4.0,0.0,0.0,0.0,3.0,3.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
Heat (1995),0.0,0.0,0.0,3.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,5.0,0.0,0.0
Jumanji (1995),3.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,4.0


In [120]:
df3_p= df2.corr(method='pearson')
df3_pe = pd.DataFrame(df3_p,columns=df.userId.unique(),index=df.userId.unique())
df3_pe.head()

,3,6,8,10,11,12,13,14,16,19,...,6975,6979,6993,7030,7031,7044,7070,7080,7087,7105
3,1.000000,0.601090,0.398015,1.000000,0.861858,0.532397,0.645497,0.763414,0.666667,0.441016,...,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111
6,0.601090,1.000000,0.344316,0.601090,0.444362,0.762855,0.220217,0.892957,0.292422,0.775857,...,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194,-0.211194
8,0.398015,0.344316,1.000000,0.398015,0.568711,0.508565,0.385376,0.160137,0.472643,0.522346,...,0.398015,0.398015,0.398015,0.398015,0.398015,0.398015,0.398015,0.398015,0.398015,0.398015
10,1.000000,0.601090,0.398015,1.000000,0.861858,0.532397,0.645497,0.763414,0.666667,0.441016,...,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111,-0.111111
11,0.861858,0.444362,0.568711,0.861858,1.000000,0.326025,0.812530,0.617008,0.527321,0.267162,...,0.408248,0.408248,0.408248,0.408248,0.408248,0.408248,0.408248,0.408248,0.408248,0.408248


* To get our output with the columns index we are converting it to the DataFrame 

In [121]:
user = 14 
thr = 0.9

In [123]:
df3_pe[df3_pe[user]>thr][user].sort_values(ascending=False)[1:11]

2554    0.999884
6823    0.999884
6426    0.999884
6128    0.999884
5338    0.999884
5035    0.999884
4476    0.999884
307     0.999884
3537    0.999884
778     0.999809
Name: 14, dtype: float64